# Updated benchmark workflow with multiple evaluation windows

This notebook updates the benchmark analysis after expanding the crypto universe from 10 to 15 assets.

It contains two benchmark parts:

1. **TDA vs market / volatility indicators**
   - Merge the 24 TDA result files into a wide panel.
   - Merge benchmark indicators:
     - Fear & Greed Index
     - VIX
     - BTC Parkinson volatility
     - BTC Rogers volatility
     - S&P 500 rolling volatility
     - BVOL
     - CVI
   - Evaluate all indicators with ROC-AUC under multiple directed evaluation windows.

2. **TDA vs raw log-return change-point baselines**
   - Use the same 15-coin log-return data that underlies the TDA construction.
   - Apply `PELT`, `BinSeg`, and `CUSUM` to both:
     - each TDA indicator series, and
     - each raw cryptocurrency log-return series.
   - Evaluate their extreme-event monitoring performance under multiple directed evaluation windows.

Evaluation windows used in both parts:
- `[-14, 0]`
- `[-7, 0]`
- `[-21, 0]`
- `[0, +7]`
- `[0, +21]`


In [ ]:
from pathlib import Path
import re
import time

import numpy as np
import pandas as pd
import ruptures as rpt
from sklearn.metrics import roc_curve, auc

TDA_DIR = Path('/Users/jane/Documents/202511吾-Systems/8. New data/15coin_tda_csv')
LOGRET_PATH = Path('/Users/jane/Documents/202511吾-Systems/8. New data/15coin_log_return_embeddings/cmc15_log_returns.csv')
MARKET_PATH = Path('/Users/jane/Documents/202511吾-Systems/8. New data/0321股票指数与金融恐慌指数_新增BVOL_CVI(CVI末端缺失).csv')
BTC_VOL_PATH = Path('/Users/jane/Documents/202511吾-Systems/8. New data/CMC_Volatility_Parkinson_Rogers.xlsx')
FNG_PATH = Path('/Users/jane/Documents/202511吾-Systems/3.Data/fear_and_greed_index.csv')
OUTPUT_DIR = Path('/Users/jane/Documents/202511吾-Systems/8. New data/benchmark_multiwindow_15coin')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

tda_files = sorted([p for p in TDA_DIR.glob('cmc15_tda_*.csv') if p.name != 'cmc15_tda_manifest.csv'])
print(f'TDA files found: {len(tda_files)}')
print(f'Output directory: {OUTPUT_DIR}')


TDA files found: 24
Output directory: /Users/jane/Documents/202511吾-Systems/8. New data/benchmark_multiwindow_15coin


In [ ]:
EVENTS = pd.to_datetime([
    '2020-12-01','2021-01-02','2021-01-07','2021-01-29','2021-02-16','2021-03-13',
    '2021-04-10','2021-05-12','2021-05-17','2021-05-18','2021-05-19','2021-06-09',
    '2021-09-24','2021-10-15','2021-11-15',
    '2022-04-27','2022-05-01','2022-05-11','2022-05-12','2022-05-13','2022-07-20',
    '2022-11-01','2023-03-01','2023-03-10','2023-05-17','2023-06-16','2023-07-01',
    '2023-10-01','2024-03-19','2024-04-20','2025-01-20','2025-02-03','2025-02-21',
    '2025-03-07','2025-05-20','2025-06-05','2025-06-17'
])

EVAL_WINDOWS = [
    ('[-14,0]', -14, 0),
    ('[-7,0]', -7, 0),
    ('[-21,0]', -21, 0),
    ('[0,+7]', 0, 7),
    ('[0,+21]', 0, 21),
]

pd.DataFrame(EVAL_WINDOWS, columns=['window_label', 'start_offset', 'end_offset']).to_csv(
    OUTPUT_DIR / 'cmc15_evaluation_windows.csv', index=False
)
pd.DataFrame({'event_date': EVENTS}).to_csv(OUTPUT_DIR / 'cmc15_extreme_events.csv', index=False)

print('Saved evaluation window and event reference tables.')


Saved evaluation window and event reference tables.


In [ ]:
tda_pattern = re.compile(r'cmc15_tda_(wasserstein|dtw)_m(\d+)_tau(\d+)\.csv')


def parse_tda_filename(path):
    m = tda_pattern.match(path.name)
    if not m:
        raise ValueError(f'Unexpected TDA filename: {path.name}')
    return m.group(1), int(m.group(2)), int(m.group(3))


tda_wide = None
indicator_manifest_rows = []

for path in tda_files:
    method, window_size, lag = parse_tda_filename(path)
    df = pd.read_csv(path, parse_dates=['date'])
    prefix = f'tda_{method}_m{window_size:02d}_tau{lag}'

    rename_map = {
        'betti0': f'{prefix}_betti0',
        'betti1': f'{prefix}_betti1',
        'persistent_entropy': f'{prefix}_entropy',
    }

    part = df[['date', 'betti0', 'betti1', 'persistent_entropy']].rename(columns=rename_map)

    if tda_wide is None:
        tda_wide = part.copy()
    else:
        tda_wide = tda_wide.merge(part, on='date', how='outer')

    for feature_name, indicator_col in [('betti0', rename_map['betti0']), ('betti1', rename_map['betti1']), ('entropy', rename_map['persistent_entropy'])]:
        indicator_manifest_rows.append({
            'indicator': indicator_col,
            'category': 'tda',
            'distance_method': method,
            'embedding_window_size': window_size,
            'embedding_lag': lag,
            'feature': feature_name,
            'source_file': path.name,
        })

tda_wide = tda_wide.sort_values('date').reset_index(drop=True)
tda_wide.to_csv(OUTPUT_DIR / 'cmc15_tda_wide_panel.csv', index=False)

indicator_manifest = pd.DataFrame(indicator_manifest_rows)
print('TDA wide shape:', tda_wide.shape)
print(indicator_manifest.head())


TDA wide shape: (1867, 73)
                  indicator category distance_method  embedding_window_size  \
0   tda_dtw_m07_tau1_betti0      tda             dtw                      7   
1   tda_dtw_m07_tau1_betti1      tda             dtw                      7   
2  tda_dtw_m07_tau1_entropy      tda             dtw                      7   
3   tda_dtw_m07_tau2_betti0      tda             dtw                      7   
4   tda_dtw_m07_tau2_betti1      tda             dtw                      7   

   embedding_lag  feature                 source_file  
0              1   betti0  cmc15_tda_dtw_m07_tau1.csv  
1              1   betti1  cmc15_tda_dtw_m07_tau1.csv  
2              1  entropy  cmc15_tda_dtw_m07_tau1.csv  
3              2   betti0  cmc15_tda_dtw_m07_tau2.csv  
4              2   betti1  cmc15_tda_dtw_m07_tau2.csv  


In [ ]:
market_df = pd.read_csv(MARKET_PATH)
market_df['date'] = pd.to_datetime(market_df['Date'])
market_df = market_df[['date', 'SP500', 'VIX', 'BVOL', 'CVI']].sort_values('date').reset_index(drop=True)
last_valid_cvi = market_df.loc[market_df['CVI'].notna(), 'date'].max()

btc_df = pd.read_excel(BTC_VOL_PATH, sheet_name='Volatility')
btc_df['date'] = pd.to_datetime(btc_df['Date'])
btc_df = btc_df[['date', 'BTC_Parkinson', 'BTC_Rogers']].rename(columns={
    'BTC_Parkinson': 'btc_parkinson',
    'BTC_Rogers': 'btc_rogers'
}).sort_values('date').reset_index(drop=True)

fng_df = pd.read_csv(FNG_PATH)
fng_df['date'] = pd.to_datetime(fng_df['Date'])
fng_df = fng_df[['date', 'fng_value']].rename(columns={'fng_value': 'fear_greed'}).sort_values('date').reset_index(drop=True)

benchmark_panel = (
    tda_wide
    .merge(market_df, on='date', how='left')
    .merge(btc_df, on='date', how='left')
    .merge(fng_df, on='date', how='left')
    .sort_values('date')
    .reset_index(drop=True)
)

# Fill weekday market indicators across weekends / non-trading days.
for col in ['SP500', 'VIX', 'BVOL']:
    benchmark_panel[col] = benchmark_panel[col].ffill()

# For CVI, keep the tail after the last valid observation as missing, but fill internal / weekend gaps.
benchmark_panel['CVI'] = benchmark_panel['CVI'].ffill()
benchmark_panel.loc[benchmark_panel['date'] > last_valid_cvi, 'CVI'] = np.nan

# S&P 500 rolling volatility benchmark.
benchmark_panel['sp500_return'] = benchmark_panel['SP500'].pct_change()
benchmark_panel['sp500_vol7'] = benchmark_panel['sp500_return'].rolling(7).std()

benchmark_indicators = ['fear_greed', 'VIX', 'btc_parkinson', 'btc_rogers', 'sp500_vol7', 'BVOL', 'CVI']

benchmark_rows = []
for col in benchmark_indicators:
    benchmark_rows.append({
        'indicator': col,
        'category': 'benchmark',
        'distance_method': np.nan,
        'embedding_window_size': np.nan,
        'embedding_lag': np.nan,
        'feature': col,
        'source_file': 'merged_benchmark_sources'
    })

indicator_manifest = pd.concat([indicator_manifest, pd.DataFrame(benchmark_rows)], ignore_index=True)
indicator_manifest.to_csv(OUTPUT_DIR / 'cmc15_indicator_manifest.csv', index=False)
benchmark_panel.to_csv(OUTPUT_DIR / 'cmc15_benchmark_panel.csv', index=False)

print('Benchmark panel shape:', benchmark_panel.shape)
print('Last valid CVI date:', last_valid_cvi)
benchmark_panel[['date'] + benchmark_indicators].head(10)


Benchmark panel shape: (1867, 82)
Last valid CVI date: 2024-11-27 00:00:00


,date,fear_greed,VIX,btc_parkinson,btc_rogers,sp500_vol7,BVOL,CVI
0,2020-08-17,83.0,21.350000,0.027459,0.024584,NaN,40.26,85.7554
1,2020-08-18,82.0,21.510000,0.018851,0.016383,NaN,40.50,85.8771
2,2020-08-19,80.0,22.540001,0.017301,0.015075,NaN,42.96,81.9934
3,2020-08-20,75.0,22.719999,0.009684,0.009182,NaN,43.01,76.9311
4,2020-08-21,81.0,22.540001,0.017113,0.010493,NaN,43.15,75.7958
5,2020-08-22,78.0,22.540001,0.012490,0.015649,NaN,43.15,75.7958
6,2020-08-23,76.0,22.540001,0.007923,0.010253,NaN,43.15,75.7958
7,2020-08-24,78.0,22.370001,0.009452,0.008905,0.004419,44.06,73.7010
8,2020-08-25,75.0,22.030001,0.030780,0.028445,0.004457,43.16,73.7044
9,2020-08-26,76.0,23.270000,0.012264,0.012460,0.004228,43.21,72.5659


In [ ]:
for window_label, start_offset, end_offset in EVAL_WINDOWS:
    label_col = f'event_label_{window_label}'
    benchmark_panel[label_col] = 0
    for event_date in EVENTS:
        start_date = event_date + pd.Timedelta(days=start_offset)
        end_date = event_date + pd.Timedelta(days=end_offset)
        benchmark_panel.loc[(benchmark_panel['date'] >= start_date) & (benchmark_panel['date'] <= end_date), label_col] = 1

benchmark_panel.to_csv(OUTPUT_DIR / 'cmc15_benchmark_panel_with_labels.csv', index=False)
label_cols = [f'event_label_{w[0]}' for w in EVAL_WINDOWS]
benchmark_panel[['date'] + label_cols].head(20)


,date,"event_label_[-14,0]","event_label_[-7,0]","event_label_[-21,0]","event_label_[0,+7]","event_label_[0,+21]"
0,2020-08-17,0,0,0,0,0
1,2020-08-18,0,0,0,0,0
2,2020-08-19,0,0,0,0,0
3,2020-08-20,0,0,0,0,0
4,2020-08-21,0,0,0,0,0
5,2020-08-22,0,0,0,0,0
6,2020-08-23,0,0,0,0,0
7,2020-08-24,0,0,0,0,0
8,2020-08-25,0,0,0,0,0
9,2020-08-26,0,0,0,0,0


In [ ]:
tda_indicators = indicator_manifest.loc[indicator_manifest['category'] == 'tda', 'indicator'].tolist()
all_auc_indicators = tda_indicators + benchmark_indicators


def evaluable_event_count(series_dates, events, start_offset, end_offset):
    series_start = pd.to_datetime(series_dates.min())
    series_end = pd.to_datetime(series_dates.max())
    count = 0
    for e in events:
        ws = e + pd.Timedelta(days=start_offset)
        we = e + pd.Timedelta(days=end_offset)
        if ws >= series_start and we <= series_end:
            count += 1
    return count


auc_rows = []

for window_label, start_offset, end_offset in EVAL_WINDOWS:
    label_col = f'event_label_{window_label}'
    y_full = benchmark_panel[label_col].to_numpy()

    for indicator in all_auc_indicators:
        y_score = pd.to_numeric(benchmark_panel[indicator], errors='coerce').to_numpy()
        mask = np.isfinite(y_score)

        if mask.sum() == 0:
            continue

        y_true = y_full[mask]
        y_score_masked = y_score[mask]

        if np.unique(y_true).size < 2:
            continue

        fpr, tpr, _ = roc_curve(y_true, y_score_masked)
        roc_auc = auc(fpr, tpr)

        series_dates = benchmark_panel.loc[mask, 'date']
        meta = indicator_manifest.loc[indicator_manifest['indicator'] == indicator].iloc[0].to_dict()

        auc_rows.append({
            'window_label': window_label,
            'start_offset': start_offset,
            'end_offset': end_offset,
            'indicator': indicator,
            'category': meta['category'],
            'distance_method': meta['distance_method'],
            'embedding_window_size': meta['embedding_window_size'],
            'embedding_lag': meta['embedding_lag'],
            'feature': meta['feature'],
            'auc': roc_auc,
            'n_obs': int(mask.sum()),
            'n_positive': int(y_true.sum()),
            'n_events_in_range': int(evaluable_event_count(series_dates, EVENTS, start_offset, end_offset)),
        })

auc_df = pd.DataFrame(auc_rows).sort_values(['window_label', 'auc'], ascending=[True, False]).reset_index(drop=True)
auc_df.to_csv(OUTPUT_DIR / 'cmc15_auc_multiwindow_long.csv', index=False)

auc_pivot = auc_df.pivot_table(index=['indicator', 'category', 'distance_method', 'embedding_window_size', 'embedding_lag', 'feature'], columns='window_label', values='auc')
auc_pivot['mean_auc'] = auc_pivot.mean(axis=1)
auc_pivot = auc_pivot.sort_values('mean_auc', ascending=False).reset_index()
auc_pivot.to_csv(OUTPUT_DIR / 'cmc15_auc_multiwindow_pivot.csv', index=False)

auc_best = auc_df.sort_values(['window_label', 'auc'], ascending=[True, False]).groupby('window_label', as_index=False).first()
auc_best.to_csv(OUTPUT_DIR / 'cmc15_auc_best_by_window.csv', index=False)

print('AUC result rows:', len(auc_df))
auc_df.head(10)


AUC result rows: 395


,window_label,start_offset,end_offset,indicator,category,distance_method,embedding_window_size,embedding_lag,feature,auc,n_obs,n_positive,n_events_in_range
0,"[-14,0]",-14,0,tda_dtw_m07_tau1_betti0,tda,dtw,7.0,1.0,betti0,0.636218,1867,452,37
1,"[-14,0]",-14,0,tda_dtw_m28_tau3_betti0,tda,dtw,28.0,3.0,betti0,0.634017,1792,452,37
2,"[-14,0]",-14,0,tda_dtw_m14_tau1_betti0,tda,dtw,14.0,1.0,betti0,0.629199,1860,452,37
3,"[-14,0]",-14,0,tda_wasserstein_m07_tau1_betti0,tda,wasserstein,7.0,1.0,betti0,0.628166,1867,452,37
4,"[-14,0]",-14,0,tda_wasserstein_m28_tau3_betti0,tda,wasserstein,28.0,3.0,betti0,0.624743,1792,452,37
5,"[-14,0]",-14,0,CVI,benchmark,NaN,NaN,NaN,CVI,0.622806,1564,352,30
6,"[-14,0]",-14,0,tda_dtw_m07_tau2_betti0,tda,dtw,7.0,2.0,betti0,0.622726,1861,452,37
7,"[-14,0]",-14,0,tda_wasserstein_m07_tau2_betti0,tda,wasserstein,7.0,2.0,betti0,0.621259,1861,452,37
8,"[-14,0]",-14,0,tda_wasserstein_m14_tau1_betti0,tda,wasserstein,14.0,1.0,betti0,0.619965,1860,452,37
9,"[-14,0]",-14,0,tda_dtw_m21_tau1_betti0,tda,dtw,21.0,1.0,betti0,0.617980,1853,452,37


In [ ]:
def detect_cps(series, method, n_bkps=10):
    x = series.astype(float).reshape(-1, 1)
    n = len(x)
    pen_val = np.log(n)

    if method == 'PELT':
        algo = rpt.Pelt(model='rbf').fit(x)
        cps = algo.predict(pen=pen_val)
    elif method == 'BinSeg':
        algo = rpt.Binseg(model='rbf').fit(x)
        cps = algo.predict(n_bkps=n_bkps)
    elif method == 'CUSUM':
        algo = rpt.KernelCPD(kernel='linear').fit(x)
        cps = algo.predict(n_bkps=n_bkps)
    else:
        raise ValueError(f'Unknown method: {method}')

    return cps[:-1]


def filter_events_for_series(events, series_start, series_end, start_offset, end_offset):
    filtered = []
    for e in events:
        ws = e + pd.Timedelta(days=start_offset)
        we = e + pd.Timedelta(days=end_offset)
        if ws >= series_start and we <= series_end:
            filtered.append(e)
    return pd.to_datetime(filtered)


def evaluate_change_points(cp_dates, events, start_offset, end_offset):
    cp_dates = sorted(pd.to_datetime(cp_dates))
    events = pd.to_datetime(events)

    if len(events) == 0:
        return {
            'n_events': 0,
            'matched_events': 0,
            'n_cp': len(cp_dates),
            'matched_cp': 0,
            'precision': np.nan,
            'recall': np.nan,
            'f1': np.nan,
        }

    windows = [(e + pd.Timedelta(days=start_offset), e + pd.Timedelta(days=end_offset)) for e in events]

    matched_events = 0
    for ws, we in windows:
        if any((cp >= ws) and (cp <= we) for cp in cp_dates):
            matched_events += 1

    matched_cp = 0
    for cp in cp_dates:
        if any((cp >= ws) and (cp <= we) for ws, we in windows):
            matched_cp += 1

    precision = matched_cp / len(cp_dates) if len(cp_dates) > 0 else 0.0
    recall = matched_events / len(events) if len(events) > 0 else np.nan

    if np.isnan(recall) or (precision + recall == 0):
        f1 = np.nan if np.isnan(recall) else 0.0
    else:
        f1 = 2 * precision * recall / (precision + recall)

    return {
        'n_events': int(len(events)),
        'matched_events': int(matched_events),
        'n_cp': int(len(cp_dates)),
        'matched_cp': int(matched_cp),
        'precision': float(precision),
        'recall': float(recall),
        'f1': float(f1),
    }


In [ ]:
logret_df = pd.read_csv(LOGRET_PATH)
logret_df['date'] = pd.to_datetime(logret_df['Date'])
logret_df = logret_df.drop(columns=['Date']).sort_values('date').reset_index(drop=True)
coin_cols = [c for c in logret_df.columns if c != 'date']

cp_series_manifest = indicator_manifest.loc[
    indicator_manifest['category'] == 'tda',
    ['indicator', 'category', 'distance_method', 'embedding_window_size', 'embedding_lag', 'feature', 'source_file']
].copy()

cp_logret_rows = []
for coin in coin_cols:
    cp_logret_rows.append({
        'indicator': f'log_return_{coin}',
        'category': 'log_return',
        'distance_method': np.nan,
        'embedding_window_size': np.nan,
        'embedding_lag': np.nan,
        'feature': coin,
        'source_file': 'cmc15_log_returns.csv',
    })

cp_series_manifest = pd.concat([cp_series_manifest, pd.DataFrame(cp_logret_rows)], ignore_index=True)
cp_series_manifest.to_csv(OUTPUT_DIR / 'cmc15_changepoint_series_manifest.csv', index=False)

logret_panel = logret_df.rename(columns={coin: f'log_return_{coin}' for coin in coin_cols})
cp_panel = tda_wide.merge(logret_panel, on='date', how='outer').sort_values('date').reset_index(drop=True)

cp_raw_rows = []
cp_eval_rows = []
cp_methods = ['PELT', 'BinSeg', 'CUSUM']
cp_start = time.time()

for idx, indicator in enumerate(cp_series_manifest['indicator'], start=1):
    meta = cp_series_manifest.loc[cp_series_manifest['indicator'] == indicator].iloc[0].to_dict()
    sub = cp_panel[['date', indicator]].dropna().sort_values('date').reset_index(drop=True)

    if len(sub) < 30:
        continue

    series = sub[indicator].to_numpy(dtype=float)
    series_start = sub['date'].min()
    series_end = sub['date'].max()

    for method in cp_methods:
        cps = detect_cps(series, method=method, n_bkps=10)
        cp_dates = [sub['date'].iloc[c] for c in cps if c < len(sub)]

        for cp_idx, cp_date in zip(cps, cp_dates):
            cp_raw_rows.append({
                'indicator': indicator,
                'category': meta['category'],
                'distance_method': meta['distance_method'],
                'embedding_window_size': meta['embedding_window_size'],
                'embedding_lag': meta['embedding_lag'],
                'feature': meta['feature'],
                'cp_method': method,
                'cp_index': int(cp_idx),
                'cp_date': pd.to_datetime(cp_date),
            })

        for window_label, start_offset, end_offset in EVAL_WINDOWS:
            filtered_events = filter_events_for_series(EVENTS, series_start, series_end, start_offset, end_offset)
            scores = evaluate_change_points(cp_dates, filtered_events, start_offset, end_offset)

            cp_eval_rows.append({
                'indicator': indicator,
                'category': meta['category'],
                'distance_method': meta['distance_method'],
                'embedding_window_size': meta['embedding_window_size'],
                'embedding_lag': meta['embedding_lag'],
                'feature': meta['feature'],
                'cp_method': method,
                'window_label': window_label,
                'start_offset': start_offset,
                'end_offset': end_offset,
                **scores,
                'series_start': series_start,
                'series_end': series_end,
            })

    if idx % 10 == 0 or idx == len(cp_series_manifest):
        print(f'Processed {idx}/{len(cp_series_manifest)} changepoint series after {time.time() - cp_start:.1f}s')

cp_raw_df = pd.DataFrame(cp_raw_rows).sort_values(['category', 'indicator', 'cp_method', 'cp_date']).reset_index(drop=True)
cp_eval_df = pd.DataFrame(cp_eval_rows).sort_values(['window_label', 'cp_method', 'f1'], ascending=[True, True, False]).reset_index(drop=True)

cp_raw_df.to_csv(OUTPUT_DIR / 'cmc15_changepoints_raw.csv', index=False)
cp_eval_df.to_csv(OUTPUT_DIR / 'cmc15_changepoint_eval_multiwindow_long.csv', index=False)

cp_pivot = cp_eval_df.pivot_table(
    index=['indicator', 'category', 'distance_method', 'embedding_window_size', 'embedding_lag', 'feature', 'cp_method'],
    columns='window_label',
    values='f1'
)
cp_pivot['mean_f1'] = cp_pivot.mean(axis=1)
cp_pivot = cp_pivot.sort_values('mean_f1', ascending=False).reset_index()
cp_pivot.to_csv(OUTPUT_DIR / 'cmc15_changepoint_eval_multiwindow_pivot.csv', index=False)

cp_best = cp_eval_df.sort_values(['window_label', 'cp_method', 'f1'], ascending=[True, True, False]).groupby(['window_label', 'cp_method'], as_index=False).first()
cp_best.to_csv(OUTPUT_DIR / 'cmc15_changepoint_best_by_window_method.csv', index=False)

cp_best_by_category = cp_eval_df.sort_values(['window_label', 'cp_method', 'category', 'f1'], ascending=[True, True, True, False]).groupby(['window_label', 'cp_method', 'category'], as_index=False).first()
cp_best_by_category.to_csv(OUTPUT_DIR / 'cmc15_changepoint_best_by_window_method_category.csv', index=False)

cp_group_summary = (
    cp_eval_df
    .groupby(['window_label', 'cp_method', 'category'], as_index=False)
    .agg(
        n_series=('indicator', 'nunique'),
        mean_f1=('f1', 'mean'),
        median_f1=('f1', 'median'),
        max_f1=('f1', 'max'),
        mean_precision=('precision', 'mean'),
        mean_recall=('recall', 'mean'),
    )
    .sort_values(['window_label', 'cp_method', 'max_f1'], ascending=[True, True, False])
    .reset_index(drop=True)
)
cp_group_summary.to_csv(OUTPUT_DIR / 'cmc15_changepoint_group_summary.csv', index=False)

print('Raw changepoint rows:', len(cp_raw_df))
print('Evaluation rows:', len(cp_eval_df))
print(cp_eval_df['category'].value_counts(dropna=False).to_string())
cp_eval_df.head(10)


Processed 10/87 changepoint series after 54.5s


Processed 20/87 changepoint series after 110.5s


Processed 30/87 changepoint series after 148.0s


Processed 40/87 changepoint series after 188.6s


Processed 50/87 changepoint series after 252.1s


Processed 60/87 changepoint series after 308.6s


Processed 70/87 changepoint series after 352.5s


Processed 80/87 changepoint series after 399.6s


Processed 87/87 changepoint series after 431.6s
Raw changepoint rows: 2415
Evaluation rows: 1305
category
tda           1080
log_return     225


,indicator,category,distance_method,embedding_window_size,embedding_lag,feature,cp_method,window_label,start_offset,end_offset,n_events,matched_events,n_cp,matched_cp,precision,recall,f1,series_start,series_end
0,tda_dtw_m07_tau2_entropy,tda,dtw,7.0,2.0,entropy,BinSeg,"[-14,0]",-14,0,37,9,10,5,0.5,0.243243,0.327273,2020-08-23,2025-09-26
1,log_return_IOTA,log_return,NaN,NaN,NaN,IOTA,BinSeg,"[-14,0]",-14,0,37,8,10,6,0.6,0.216216,0.317881,2020-08-11,2025-09-26
2,tda_wasserstein_m21_tau3_betti0,tda,wasserstein,21.0,3.0,betti0,BinSeg,"[-14,0]",-14,0,37,8,10,5,0.5,0.216216,0.301887,2020-10-10,2025-09-26
3,tda_wasserstein_m28_tau2_entropy,tda,wasserstein,28.0,2.0,entropy,BinSeg,"[-14,0]",-14,0,37,8,10,5,0.5,0.216216,0.301887,2020-10-04,2025-09-26
4,tda_wasserstein_m21_tau1_betti0,tda,wasserstein,21.0,1.0,betti0,BinSeg,"[-14,0]",-14,0,37,7,10,4,0.4,0.189189,0.256881,2020-08-31,2025-09-26
5,tda_wasserstein_m28_tau1_betti0,tda,wasserstein,28.0,1.0,betti0,BinSeg,"[-14,0]",-14,0,37,7,10,4,0.4,0.189189,0.256881,2020-09-07,2025-09-26
6,tda_dtw_m14_tau2_betti0,tda,dtw,14.0,2.0,betti0,BinSeg,"[-14,0]",-14,0,37,6,10,4,0.4,0.162162,0.230769,2020-09-06,2025-09-26
7,tda_wasserstein_m14_tau2_betti0,tda,wasserstein,14.0,2.0,betti0,BinSeg,"[-14,0]",-14,0,37,6,10,4,0.4,0.162162,0.230769,2020-09-06,2025-09-26
8,log_return_ETH,log_return,NaN,NaN,NaN,ETH,BinSeg,"[-14,0]",-14,0,37,6,10,4,0.4,0.162162,0.230769,2020-08-11,2025-09-26
9,log_return_DOGE,log_return,NaN,NaN,NaN,DOGE,BinSeg,"[-14,0]",-14,0,37,5,10,5,0.5,0.135135,0.212766,2020-08-11,2025-09-26


In [ ]:
print('Saved files in output directory:')
for p in sorted(OUTPUT_DIR.glob('*')):
    print('-', p.name)

print('\nTop AUC rows:')
display(auc_df.head(5))

print('\nTop changepoint rows:')
display(cp_eval_df.head(5))


Saved files in output directory:
- cmc15_auc_best_by_window.csv
- cmc15_auc_multiwindow_long.csv
- cmc15_auc_multiwindow_pivot.csv
- cmc15_benchmark_panel.csv
- cmc15_benchmark_panel_with_labels.csv
- cmc15_changepoint_best_by_window_method.csv
- cmc15_changepoint_best_by_window_method_category.csv
- cmc15_changepoint_eval_multiwindow_long.csv
- cmc15_changepoint_eval_multiwindow_long_test.csv
- cmc15_changepoint_eval_multiwindow_pivot.csv
- cmc15_changepoint_group_summary.csv
- cmc15_changepoint_series_manifest.csv
- cmc15_changepoints_raw.csv
- cmc15_evaluation_windows.csv
- cmc15_extreme_events.csv
- cmc15_indicator_manifest.csv
- cmc15_tda_wide_panel.csv

Top AUC rows:


,window_label,start_offset,end_offset,indicator,category,distance_method,embedding_window_size,embedding_lag,feature,auc,n_obs,n_positive,n_events_in_range
0,"[-14,0]",-14,0,tda_dtw_m07_tau1_betti0,tda,dtw,7.0,1.0,betti0,0.636218,1867,452,37
1,"[-14,0]",-14,0,tda_dtw_m28_tau3_betti0,tda,dtw,28.0,3.0,betti0,0.634017,1792,452,37
2,"[-14,0]",-14,0,tda_dtw_m14_tau1_betti0,tda,dtw,14.0,1.0,betti0,0.629199,1860,452,37
3,"[-14,0]",-14,0,tda_wasserstein_m07_tau1_betti0,tda,wasserstein,7.0,1.0,betti0,0.628166,1867,452,37
4,"[-14,0]",-14,0,tda_wasserstein_m28_tau3_betti0,tda,wasserstein,28.0,3.0,betti0,0.624743,1792,452,37



Top changepoint rows:


,indicator,category,distance_method,embedding_window_size,embedding_lag,feature,cp_method,window_label,start_offset,end_offset,n_events,matched_events,n_cp,matched_cp,precision,recall,f1,series_start,series_end
0,tda_dtw_m07_tau2_entropy,tda,dtw,7.0,2.0,entropy,BinSeg,"[-14,0]",-14,0,37,9,10,5,0.5,0.243243,0.327273,2020-08-23,2025-09-26
1,log_return_IOTA,log_return,NaN,NaN,NaN,IOTA,BinSeg,"[-14,0]",-14,0,37,8,10,6,0.6,0.216216,0.317881,2020-08-11,2025-09-26
2,tda_wasserstein_m21_tau3_betti0,tda,wasserstein,21.0,3.0,betti0,BinSeg,"[-14,0]",-14,0,37,8,10,5,0.5,0.216216,0.301887,2020-10-10,2025-09-26
3,tda_wasserstein_m28_tau2_entropy,tda,wasserstein,28.0,2.0,entropy,BinSeg,"[-14,0]",-14,0,37,8,10,5,0.5,0.216216,0.301887,2020-10-04,2025-09-26
4,tda_wasserstein_m21_tau1_betti0,tda,wasserstein,21.0,1.0,betti0,BinSeg,"[-14,0]",-14,0,37,7,10,4,0.4,0.189189,0.256881,2020-08-31,2025-09-26
